In [20]:
import pandas as pd
import geopandas as gpd
import numpy as np
import mapclassify
import folium
import os

In [21]:
FOLDER = r'C:\Users\Roberto Ponce López\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey (1)\Modelación Urbana - Red Vial Guadalajara'
SHP_FOLDER = "red_shapefiles"
VISUM_FOLDER = "Visum Objects"
NETWORK_FOLDER = "Red Base GDL"

CRS_ORIGINAL = "EPSG:4326"
CRS_METROS = "EPSG:32613"

### 1) Buffer around zones

In [30]:
ZONE_BUFFER = 10 # 10 metros around zone

zones_visum = gpd.read_file(os.path.join(FOLDER, SHP_FOLDER, "zones_agebs_visum", "zonas_agebs_standarized.shp"))

no_tala_mask = (
    (zones_visum["id_mun_age"] < 4000) | (zones_visum["id_mun_age"] > 4999)
)

#project to metros to apply the buffer
zones_visum_m = zones_visum.to_crs(CRS_METROS)
zones_visum_m["buffer_10m"] = zones_visum_m.geometry.buffer(ZONE_BUFFER)
zones_buffers = zones_visum_m.set_geometry("buffer_10m")

In [31]:
zones_buffers_plot = zones_buffers.to_crs(CRS_ORIGINAL)

m = zones_buffers_plot.explore(
    color="red",
    style_kwds={"fillOpacity": 0.3},
    tiles="OpenStreetMap"
)

zones_visum.explore(
    m=m,
    color="blue",
    style_kwds={"fillOpacity": 0.4}
)

### 2) Which relevant PuT nodes interest zone buffer?
- __relevant PuT nodes:__ nodes with a SP or with a link (in|out) that has a SP

In [32]:
nodes = gpd.read_file(os.path.join(FOLDER, VISUM_FOLDER, "Nodes", "nodes for put connectors", "nodes.shp"))
nodes = nodes.drop(columns=["XCoord", "YCoord"])
nodes = nodes.rename(columns={"INLINK_HAS": "INLINK_HAS_SP", "OUTLINK_HA": "OUTLINK_HAS_SP"})

# get relevant put nodes
relevant_put_nodes = (
    (nodes["N_SP"] > 0) |
    (nodes["INLINK_HAS_SP"] != "0") |
    (nodes["OUTLINK_HAS_SP"] != "0")
)
put_nodes = nodes[relevant_put_nodes].copy()

# reproject to metros crs
put_nodes_m = put_nodes.to_crs(CRS_METROS)
print(f"{len(put_nodes_m):,} nodos que tienen SP o cuyo link (de entrada o salida) tiene SP")

21,038 nodos que tienen SP o cuyo link (de entrada o salida) tiene SP


In [33]:
put_nodes.dtypes

No                 float64
N_SP               float64
INLINK_HAS_SP          str
OUTLINK_HAS_SP         str
geometry          geometry
dtype: object

In [34]:
# qué stop points intersectan con los buffers de las zonas
put_nodes_in_buffers = gpd.sjoin(
    put_nodes_m, 
    zones_buffers[["id_mun_age", "buffer_10m"]],
    how="inner",
    predicate="intersects"
)
put_nodes_in_buffers

print(f"Hay {len(zones_visum):,} zonas en visum.")
print(f"De las cuales sólo analizaremos las {zones_visum[no_tala_mask].shape[0]:,} zonas que no son de Tala.")
print(f"De esas {zones_visum[no_tala_mask].shape[0]:,} zonas, {put_nodes_in_buffers["id_mun_age"].nunique():,} tienen al menos un nodo de SP dentro de su buffer de {ZONE_BUFFER} metros.")

Hay 2,203 zonas en visum.
De las cuales sólo analizaremos las 2,096 zonas que no son de Tala.
De esas 2,096 zonas, 1,714 tienen al menos un nodo de SP dentro de su buffer de 10 metros.


In [35]:
put_connectors_per_zone = (
    put_nodes_in_buffers
    .groupby("id_mun_age")["No"]
    .nunique()
    .reset_index(name="n_put_nodes")
)
put_connectors_per_zone

,id_mun_age,n_put_nodes
0,1,15
1,2,61
2,3,24
3,4,29
4,5,9
...,...,...
1709,9046,23
1710,9047,62
1711,9048,17
1712,9049,10


In [36]:
put_connectors_per_zone["n_put_nodes"].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
)

count    1714.000000
mean       18.073512
std        15.560490
min         1.000000
25%         7.000000
50%        15.000000
75%        25.000000
90%        35.000000
95%        42.000000
99%        66.870000
max       199.000000
Name: n_put_nodes, dtype: float64

### 3) Cruzar con conectores PrT ya existentes

In [47]:
# PrT connectors (on Visum already)
prt_connectors = gpd.read_file(os.path.join(FOLDER, VISUM_FOLDER, "Connectors", "connectors.shp"))
prt_connectors = prt_connectors[["ZoneNo", "node_id"]]
prt_connectors = prt_connectors.drop_duplicates(subset=["ZoneNo", "node_id"], keep="first")

# PuT connectors (to create on Visum)
put_connectors = put_nodes_in_buffers[["id_mun_age", "No", "geometry"]].copy()
put_connectors = put_connectors.rename(columns={"id_mun_age": "ZoneNo", "No": "node_id"})

# Identificar cuáles PuT connectors ya existen como PrT connectors
put_connectors_check = put_connectors.merge(
    prt_connectors[["ZoneNo", "node_id"]],
    on=["ZoneNo", "node_id"],
    how="left",
    indicator=True
)

In [54]:
# 1. PuT connectors que NO existen como PrT → hay que crearlos
put_connectors_to_create = (
    put_connectors_check[
        put_connectors_check["_merge"] == "left_only"
    ]
    .drop(columns="_merge")
    .copy()
)
print(f"Hay {len(put_connectors_to_create):,} PuT connectors que no existen como PrT connectors y que hay que crear en Visum.")
put_connectors_to_create.to_csv(os.path.join(FOLDER, SHP_FOLDER, "PuT", "put_connectors", "put_connectors_to_create.csv"), index=False)

Hay 28,374 PuT connectors que no existen como PrT connectors y que hay que crear en Visum.


In [55]:
# 2. PuT connectors que YA existen como PrT
put_connectors_already_prt = (
    put_connectors_check[
        put_connectors_check["_merge"] == "both"
    ]
    .drop(columns="_merge")
    .copy()
)
print(f"Hay {len(put_connectors_already_prt):,} PuT connectors que ya existen como PrT connectors en Visum.")
print(f"Estos solo hay que abrirlos para TSysSet = W tambien")
put_connectors_already_prt.to_csv(os.path.join(FOLDER, SHP_FOLDER, "PuT", "put_connectors", "put_connectors_already_prt.csv"), index=False)

Hay 2,604 PuT connectors que ya existen como PrT connectors en Visum.
Estos solo hay que abrirlos para TSysSet = W tambien


### 4) Insert in Visum

In [22]:
import win32com.client as com

#Red base GDL (con 2,203 zonas)
red_base = r"C:\Users\Roberto Ponce López\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey (1)\Modelación Urbana - Red Vial Guadalajara\Red Base GDL\RedBase Conectores y Atts\RedBase 250826 - copia.ver"
Visum = com.Dispatch("Visum.Visum") #Visum 24 version
Visum.LoadVersion(red_base)
C = com.constants

In [24]:
put_connectors_already_prt = pd.read_csv(os.path.join(FOLDER, SHP_FOLDER, "PuT", "put_connectors", "put_connectors_already_prt.csv"))
put_connectors_already_prt["node_id"] = put_connectors_already_prt["node_id"].astype(int)

visum_connectors = pd.DataFrame({
    "ZoneNo": [i[1] for i in Visum.Net.Connectors.GetMultiAttValues("ZoneNo")],
    "node_id": [i[1] for i in Visum.Net.Connectors.GetMultiAttValues("NodeNo")],
})

visum_connectors = visum_connectors.merge(
    put_connectors_already_prt,
    on=["ZoneNo","node_id"],
    how="left",
    indicator=True
)

visum_connectors = visum_connectors.reset_index(drop=True)
visum_connectors.index = visum_connectors.index + 1

put_and_prt = list(zip(visum_connectors.index, visum_connectors["_merge"]))

Visum.Net.Connectors.SetMultiAttValues("PUT_AND_PRT", put_and_prt)

In [25]:
put_connectors_to_create = pd.read_csv(os.path.join(FOLDER, SHP_FOLDER, "PuT", "put_connectors", "put_connectors_to_create.csv"))
put_connectors_to_create["node_id"] = put_connectors_to_create["node_id"].astype(int)

for row in put_connectors_to_create.itertuples():
    zone_no = row.ZoneNo
    node_no = row.node_id

    try:
        # Create connector from zone to node
        
        Visum.Net.AddConnector(zone_no, node_no)

        # Get both direction connectors
        #source_connector = Visum.Net.Connectors.SourceItemByKey(zone_no, node_no)
        #destination_connector = Visum.Net.Connectors.DestItemByKey(node_no, zone_no)

        # Set TSysSet to "C" for both connectors
        #source_connector.SetAttValue("TSysSet", "C")
        #destination_connector.SetAttValue("TSysSet", "C")
    except Exception as e:
        print(f"Error creating connector for zone {zone_no} and node {node_no}: {e}")